# MuJoCo Quickstart 🤖

This notebook gets you running MuJoCo simulations in Google Colab with zero setup.

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy

print(f"MuJoCo version: {mujoco.__version__}")

## 1. Define a model in MJCF (XML)

In [ ]:
XML = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom type="plane" size="5 5 0.1" rgba=".9 .9 .9 1"/>
    <body name="ball" pos="0 0 2">
      <joint type="free"/>
      <geom type="sphere" size="0.2" rgba="1 0.3 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(XML)
data = mujoco.MjData(model)
print(f"Model has {model.nq} DOFs, timestep = {model.opt.timestep}s")

## 2. Simulate and render

In [ ]:
renderer = mujoco.Renderer(model, height=480, width=640)
frames = []
fps = 30
duration = 2.0

mujoco.mj_resetData(model, data)
data.qvel[0] = 3.0  # initial x velocity

while data.time < duration:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * fps:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

renderer.close()
mediapy.show_video(frames, fps=fps)

## 3. Add a controller

Let's control a pendulum with a PD controller.

In [ ]:
PENDULUM_XML = """
<mujoco>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <body pos="0 0 2">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -1" size="0.04" rgba="0 .7 0 1"/>
    </body>
  </worldbody>
  <actuator>
    <motor joint="hinge" ctrlrange="-10 10"/>
  </actuator>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(PENDULUM_XML)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

target = np.pi / 4  # 45 degrees
kp, kd = 50.0, 10.0
frames = []

while data.time < 5.0:
    # PD control
    data.ctrl[0] = kp * (target - data.qpos[0]) - kd * data.qvel[0]
    mujoco.mj_step(model, data)
    if len(frames) < data.time * fps:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

renderer.close()
mediapy.show_video(frames, fps=fps)

## Next Steps

- [MJCF Reference](https://mujoco.readthedocs.io/en/stable/XMLreference.html) — build your own models
- [Gymnasium MuJoCo envs](https://gymnasium.farama.org/environments/mujoco/) — RL-ready environments
- [MJX](https://mujoco.readthedocs.io/en/stable/mjx.html) — GPU-parallel simulation with JAX
- [MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie) — robot model zoo